# Stage 1 (Part 3): MiniMind LoRA 與 DPO (偏好對齊) 實戰

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EndeavorYen/LLM-learning/blob/main/notebooks/01_stage1_minimind/03_minimind_lora_dpo_colab.ipynb)

本 Notebook 帶領你完成 Stage 1 最後兩塊核心拼圖：
1. **LoRA (Low-Rank Adaptation)**：凍結原本 3,000 萬顆旋鈕，只訓練外掛的兩個低秩小矩陣 $A$ 與 $B$（檔案僅 ~2MB，100% 不破壞原知識）
2. **DPO (Direct Preference Optimization)**：同時載入「策略模型 (Policy)」與「凍結參考模型 (Reference)」，透過 `chosen`（好回答）與 `rejected`（壞回答）的機率差值進行偏好對齊

## 1. 執行 LoRA 微調 (`train_lora.py` + `lora_identity.jsonl`)
我們下載官方的自我認知資料集 `lora_identity.jsonl`（教模型認識自己叫 MiniMind），觀察 LoRA 參數佔比與訓練後的外掛檔案大小。

In [ ]:
import os, requests
os.makedirs('/content/minimind/dataset', exist_ok=True)
lora_data = '/content/minimind/dataset/lora_identity.jsonl'
if not os.path.exists(lora_data):
    url = 'https://huggingface.co/datasets/jingyaogong/minimind_dataset/resolve/main/lora_identity.jsonl'
    r = requests.get(url)
    with open(lora_data, 'wb') as f:
        f.write(r.content)

%cd /content/minimind/trainer
!python train_lora.py \
    --data_path ../dataset/lora_identity.jsonl \
    --from_weight full_sft \
    --lora_name lora_identity \
    --lora_rank 8 \
    --hidden_size 512 \
    --num_hidden_layers 8 \
    --epochs 15 \
    --batch_size 16 \
    --learning_rate 1e-4 \
    --log_interval 10

## 2. 執行 DPO 偏好對齊 (`train_dpo.py` + `dpo.jsonl`)

In [ ]:
dpo_fast = '/content/minimind/dataset/dpo_fast_500.jsonl'
if not os.path.exists(dpo_fast):
    url = 'https://huggingface.co/datasets/jingyaogong/minimind_dataset/resolve/main/dpo.jsonl'
    with requests.get(url, stream=True) as r, open(dpo_fast, 'wb') as f:
        for i, line in enumerate(r.iter_lines()):
            if i >= 500:
                break
            if line:
                f.write(line + b'\n')

%cd /content/minimind/trainer
!python train_dpo.py \
    --data_path ../dataset/dpo_fast_500.jsonl \
    --from_weight full_sft \
    --hidden_size 512 \
    --num_hidden_layers 8 \
    --epochs 1 \
    --batch_size 8 \
    --learning_rate 1e-6 \
    --beta 0.15 \
    --log_interval 15